# Modelado Predictivo: Clasificación con K-Nearest Neighbors
**Proyecto:** Clasificador de Salud Fetal (Dataset Cardiotocography - UCI).

**Contexto:** Tomando como base el dataset limpio, libre de fugas de datos ("X_features.csv" y "y_target.csv"), este notebook implementa y evalúa el modelo **K-Nearest Neighbors** construido desde cero. KNN es un algoritmo de *aprendizaje perezoso (lazy learning)* que no construye una ecuación interna, sino que memoriza el dataset de entrenamiento y clasifica a los nuevos datos basándose en la etiqueta mayoritaria de sus K vecinos más cercanos utilizando la distancia Euclidiana.

### Objetivos del Modelado

1. **Construcción y Búsqueda Vecinal:** Instanciar el clasificador KNNClassifier programado a mano con ayuda del curso 'Machine_Learning_desde_Cero-11', que calculará la distancia geométrica exacta entre un nuevo registro cardiotocográfico y los  pacientes del conjunto de entrenamiento.

2. **Evaluación de Probabilidades:** Utilizar la proporción de votos de los vecinos más cercanos para generar una distribución de probabilidad continua 'predict_proba', indispensable para el cálculo del AUC.

3. **Análisis Clínico y Costo Computacional:** Comparar las métricas de rendimiento (F1-Score, Recall, Matriz de Confusión y ROC-AUC) frente a scikit-learn, analizando la viabilidad del modelo casero en la fase de predicción debido al alto costo computacional del cálculo de distancias por fuerza bruta.


### Ficha Metodológica del Modelo

* **Descripción breve del método:** Algoritmo no paramétrico que clasifica un paciente buscando a sus K vecinos más parecidos (menor distancia Euclidiana) en el conjunto de entrenamiento. El diagnóstico final se toma por votación mayoritaria.

* **Justificación de su uso:** Es el método más intuitivo en medicina basada en evidencia: *"Si el monitor de este paciente se comporta matemáticamente igual al de otros 5 pacientes pasados que tuvieron hipoxia, lo más probable es que este paciente también la tenga"*. Dado que depende de distancias, **la estandarización previa de los datos (Z-Score) es un paso obligatorio** para evitar que variables grandes (como los latidos base LB) dominen.

* **Métricas de evaluación utilizadas:** Exactitud Global (*Accuracy*), Matriz de Confusión, *Recall*, *Precisión*, *F1-Score* por clase, *Promedio Macro F1-Score* y el cálculo del *Área Bajo la Curva ROC (AUC)*.

* **Análisis de resultados:** Detallado al final del notebook, resaltando la sensibilidad obtenida en la clase patológica.

In [1]:
# Importación de librerías 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

# Modificación para importar los modelos fuera de la carpeta actual
import os
import sys

root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_dir not in sys.path:
    sys.path.append(root_dir)

# Importación del modelo y métricas caseras
from src.Knn import KNNClassifier
from src.utils_and_metrics import train_test_split, standardize, accuracy, confusion_matrix, recall, f1_score, precision, roc

#### **1. Preparación de Datos: Partición y Estandarización Z-Score**
Para que la distancia Euclidiana funcione correctamente y no le dé un peso desproporcionado a variables con números grandes, debemos escalar las 21 características usando el método standardize. Para evitar fugas de información clínica, la media y desviación estándar se aprenden **solo** del conjunto de entrenamiento.

In [4]:
# Carga directa de matrices generadas en 02_EDA_Visualizacion
X_df = pd.read_csv('../data/processed/X_features.csv')
y_df = pd.read_csv('../data/processed/y_target.csv')

X = X_df.to_numpy()
y = y_df['NSP'].to_numpy().ravel().astype(int)

# Partición de datos (80% entrenamiento, 20% prueba) con seed=42 para comparación directa
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, seed=42)

# Estandarización Z-Score (calculando media y desviación estándar solo en X_train)
X_train, mean, std = standardize(X_train)
X_test = (X_test - mean) / std

# Visualización
print(f"Dimensión de Matriz X (Características): {X.shape}")
print(f"Dimensión de Vector y (Objetivo NSP):    {y.shape}")
print(f"Entrenamiento: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Prueba:        X_test={X_test.shape}, y_test={y_test.shape}")
print(f"Clases en y:   {np.unique(y)}")

Dimensión de Matriz X (Características): (2126, 21)
Dimensión de Vector y (Objetivo NSP):    (2126,)
Entrenamiento: X_train=(1700, 21), y_train=(1700,)
Prueba:        X_test=(426, 21), y_test=(426,)
Clases en y:   [1 2 3]


#### **2. "Entrenamiento" y Evaluación de KNN**


In [6]:
# Configuración e inicialización del modelo
n_neighbors = 5  # Número de vecinos más cercanos a consultar

knn_model = KNNClassifier(n_neighbors=n_neighbors)

t0 = time.perf_counter()

knn_model.fit(X_train, y_train)

tiempo_train_knn = time.perf_counter() - t0

# Evaluación sobre el conjunto de entrenamiento
train_preds = knn_model.predict(X_train)
train_accuracy = accuracy(y_train, train_preds)

print("\n--Resultados de Rendimiento en Entrenamiento--")
print(f"Vecinos configurados (n_neighbors): {n_neighbors}")
print(f"Accuracy Global en Entrenamiento:   {train_accuracy * 100:.2f}%")
print(f"Tiempo de entrenamiento:            {tiempo_train_knn:.4f} s")
print(matriz_test)


--Resultados de Rendimiento en Entrenamiento--
Vecinos configurados (n_neighbors): 5
Accuracy Global en Entrenamiento:   94.41%
Tiempo de entrenamiento:            0.0004 s
[[326   8   1]
 [ 25  38   2]
 [  4   4  18]]


#### **Prueba de (`KNNClassifier`)**

In [ ]:
# Predicción final sobre el conjunto de prueba
t0 = time.perf_counter()

test_preds = knn_model.predict(X_test)

tiempo_pred_knn = time.perf_counter() - t0

test_accuracy = accuracy(y_test, test_preds)

# Ajuste de etiquetas (1, 2, 3   a   0, 1, 2) para indexar la matriz de confusión
clases_unicas = np.unique(y_test)
y_real_test = (y_test - clases_unicas[0]).astype(int)
y_pred_test = (test_preds - clases_unicas[0]).astype(int)
n_classes = len(clases_unicas)

# Cálculo de matriz y métricas por clase
matriz = confusion_matrix(y_real_test, y_pred_test, n_classes)
precisions = precision(matriz)
recalls = recall(matriz)
f1s = f1_score(matriz)

print("\n--Resultado de Rendimiento en Prueba--")
print(f"Exactitud Global en Prueba:  {test_accuracy * 100:.2f}%")
print(f"Tiempo de predicción:        {tiempo_pred_knn:.4f} s")

print("\n--Métricas detalladas por Clase--")
nombres_clases = ["1 (Normal)", "2 (Sospechoso)", "3 (Patológico)"]
for i, nombre in enumerate(nombres_clases):
    print(f"Clase {nombre:14s} -> Precision: {precisions[i]*100:.2f}% | Recall: {recalls[i]*100:.2f}% | F1-Score: {f1s[i]*100:.2f}%")

print(f"\nPromedio Macro F1-Score: {np.mean(f1s)*100:.2f}%")
print("\nMatriz de Confusión (Filas: Real | Columnas: Predicho):")
print(matriz)


--Resultado de Rendimiento en Prueba--
Exactitud Global en Prueba:  89.67%
Tiempo de predicción:        5.7118 s

--Métricas detalladas por Clase--
Clase 1 (Normal)     -> Precision: 91.83% | Recall: 97.31% | F1-Score: 94.49%
Clase 2 (Sospechoso) -> Precision: 76.00% | Recall: 58.46% | F1-Score: 66.09%
Clase 3 (Patológico) -> Precision: 85.71% | Recall: 69.23% | F1-Score: 76.60%

Promedio Macro F1-Score: 79.06%

Matriz de Confusión (Filas: Real | Columnas: Predicho):
[[326   8   1]
 [ 25  38   2]
 [  4   4  18]]


#### Análisis y Conclusiones Finales del Modelo


---
#### Comparación con modelo de Scikit-learn

In [ ]:
# Modelo de ScikitLearn
from sklearn.neighbors import KNeighborsClassifier as SklearnKNN

# Ajuste y configuración equivalente a nuestro modelo
knn_sklearn = SklearnKNN(
    n_neighbors=5,
    metric="euclidean",
    n_jobs=-1,
)

t0 = time.perf_counter()
knn_sklearn.fit(X_train, y_train)
tiempo_train_sk = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_sk = knn_sklearn.predict(X_test)
tiempo_pred_sk = time.perf_counter() - t0

y_proba_sk = knn_sklearn.predict_proba(X_test)

In [ ]:
conf_matrix_sk = confusion_matrix(y_test - 1, y_pred_sk - 1, n_classes=3)
f1s_sk = f1_score(conf_matrix_sk)

# Resultados de rendimiento interno y en prueba
print("--- Rendimiento KNN (Scikit-Learn) ---")
print(f"-> Exactitud Global en Entrenamiento: {accuracy(y_train, knn_sklearn.predict(X_train)) * 100:.2f}%")
print(f"-> Exactitud Global en Prueba (Test): {accuracy(y_test, y_pred_sk) * 100:.2f}%")
print(f"-> Tiempo Entrenamiento / Predicción: {tiempo_train_sk:.4f} s / {tiempo_pred_sk:.4f} s")
print(f"-> F1-Score:                    {f1s_sk}")
print(f"-> Recall:                      {recall(conf_matrix_sk)}")
print(f"-> Precision:                   {precision(conf_matrix_sk)}")
print(f"-> Promedio Macro F1-Score:     {np.mean(f1s_sk) * 100:.2f}%")
print("\nMatriz de Confusión (Filas: Real | Columnas: Predicho):")
print(conf_matrix_sk)

In [ ]:
# Modelo Scikit-Learn
macro_auc_sk, aucs_por_clase_sk = roc(knn_sklearn, X_test, y_test)
print(f"Macro AUC Global (Scikit-Learn): {macro_auc_sk:.4f}")

In [ ]:
# Nuestro Modelo
macro_auc_knn, aucs_por_clase_knn = roc(knn_model, X_test, y_test)
print(f"Macro AUC Global (Modelo Propio): {macro_auc_knn:.4f}")

#### Conclusión de la Comparativa (Modelo Propio vs. Scikit-Learn)

